# 实验：多轮检索 Deep Research Agent

本笔记本实现了基于 ReAct 的多轮检索。它利用我们新编写的 `agent/react_agent.py`，具有以下特性：
- 真正的 `while` 循环状态机控制多轮调用。
- 利用滑动窗口处理上下文管理，防止 token 溢出。
- 满足不再硬编码假数据的要求，产生真正互动的工具调用轨迹以作评估。

## 0. 环境与服务说明

In [ ]:
!pip install -r agent/requirements.txt

: 

## 1. 基础配置

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

VLLM_BASE_URL = 'http://127.0.0.1:8000/v1'
MODEL_NAME = 'qwen_auto' # 使用 Qwen3-8B
API_KEY = 'dummy'

## 2. 初始化检索、客户端与工具注册表

In [ ]:
from agent.vllm_client import VLLMClient
from agent.tools import build_searcher, get_agent_tool_specs_and_registry
from agent.dataset_utils import load_jsonl
from agent.react_agent import run_agent_loop

bm25_index_path = str('indexes/browsecomp_plus_bm25.sqlite')
hard50_path = str(project_root / 'browsecomp_plus_hard50.jsonl')

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)
searcher = build_searcher(index_path=bm25_index_path)

# 获取工具描述与可执行注册表
tools, registry = get_agent_tool_specs_and_registry(searcher=searcher, k=5, snippet_max_chars=1200)
print('search_type:', searcher.search_type)
print('registered tools:', list(registry.keys()))

## 3. 生成轨迹文件（submission.jsonl）

In [ ]:
import json

def generate_submission(rows, output_path: str, max_turns: int = 5):
    """批量生成真实的多轮轨迹并写入 submission.jsonl。"""
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    records = []
    with output_file.open("w", encoding="utf-8") as fout:
        for i, row in enumerate(rows):
            print(f"[{i+1}/{len(rows)}] Processing query_id={row['query_id']}...")
            
            # 执行多轮循环
            answer_text, full_history = run_agent_loop(
                client=client,
                model=MODEL_NAME,
                query=row["query"],
                tools=tools,
                registry=registry,
                max_turns=max_turns,
                max_history_msgs=6 # 滑动窗口最多保留最后 6 条消息（约3回合）
            )
            
            traj = {
                "query_id": row["query_id"],
                "status": "completed",
                "predicted_answer": answer_text,
                "messages": full_history
            }
            records.append(traj)
            json.dump(traj, fout, ensure_ascii=False)
            fout.write("\n")

    print(f"\nSaved {len(records)} trajectories to: {output_path}")
    return records

submission_path = str(project_root / "runs" / "submission.jsonl")
rows = load_jsonl(hard50_path, limit=50)  # 处理全部 50 条
records = generate_submission(rows, output_path=submission_path)

print("\n--- 第一条轨迹样例 ---")
sample = records[0]
print(f"query_id: {sample['query_id']}")
print(f"status: {sample['status']}")
print(f"predicted_answer (前200字): {sample['predicted_answer'][:200]}...")
print(f"messages 条数: {len(sample['messages'])}")
for msg in sample["messages"]:
    role = msg["role"]
    has_tool_calls = "tool_calls" in msg and msg["tool_calls"]
    content_preview = str(msg.get("content", ""))[:80].replace("\n", " ")
    print(f"  [{role}] tool_calls={has_tool_calls} | {content_preview}...")

## 4. 自动评估

In [ ]:
from agent.eval import run_evaluation_with_llm

gold_dataset = load_jsonl(hard50_path)

accuracy, results = run_evaluation_with_llm(
    predictions_file=submission_path,
    gold_dataset=gold_dataset,
    client=client,
    model_name=MODEL_NAME,
)

print(f"\nOverall Accuracy: {accuracy:.2%}")